# GS200 直流电压/电流源控制示例

本示例演示如何使用 `gs200` 包控制 Yokogawa GS200 直流电压/电流源。

## 前提条件
- GS200 已通过 USB、Ethernet 或 RS-232 连接到电脑
- VISA 驱动已安装（NI-VISA 或 pyvisa-py）
- 虚拟环境 `agent_exp_env` 已配置好依赖
- 在 VSCode 中打开此 notebook 后，右上角选择内核 `Python (agent_exp_env)`

In [1]:
# 设置项目根目录
from pathlib import Path
project_root = Path.cwd().parent

In [2]:
from gs200 import GS200Instrument
import time
print("模块导入成功")

模块导入成功


---
## 1. 查找设备与建立连接

> **注意**: 以下代码需要连接 GS200 硬件才能运行。

### 查找可用资源

In [3]:
import pyvisa
rm = pyvisa.ResourceManager()
resources = rm.list_resources()
print("可用 VISA 资源:")
for res in resources:
    print(f"  {res}")
rm.close()

可用 VISA 资源:
  USB0::0x1AB1::0x0641::DG4E231500376::INSTR
  USB0::0x1AB1::0x0646::DG9Q271200104::INSTR
  USB0::0x1AB1::0x0641::DG4E222800868::INSTR
  USB0::0x1AB1::0x0641::DG4E234902522::INSTR
  USB0::0x0B21::0x0039::90Z631552::INSTR
  USB0::0x1AB1::0x0641::DG4E242401288::INSTR
  USB0::0xF4EC::0x1015::SDSEV82X900704::INSTR
  ASRL1::INSTR
  ASRL3::INSTR


### 连接设备

请根据实际接口替换 `RESOURCE` 字符串：
- USB: `USB0::0x0B21::0x0039::XXXXXXXX::INSTR`
- Ethernet: `TCPIP0::192.168.1.10::inst0::INSTR`
- RS-232: `ASRL3::INSTR` (COM3)

In [4]:
# 请替换为实际的 VISA 资源字符串
# RESOURCE = "TCPIP0::192.168.1.10::inst0::INSTR"  # Ethernet
RESOURCE =  "USB0::0x0B21::0x0039::90Z631552::INSTR"  # USB
# RESOURCE = "ASRL3::INSTR"  # RS-232

try:
    gs = GS200Instrument(RESOURCE)
    gs.connect()
    print(f"已连接: {gs.idn()}")
except Exception as e:
    print(f"连接失败: {e}")
    print("后续示例将使用模拟/脱机模式说明")

已连接: YOKOGAWA,GS210,90Z631552,2.02


---
## 2. 设置输出电流

该示例将 GS200 设为电流源模式并输出指定电流。

In [7]:
if gs.connected:
    # 设置输出电流为 9.29 mA
    # set_current 会自动切换源功能到 CURRent 模式
    gs.set_current(0.00929)  # 9.29 mA
    print(f"电流设置为: {gs.get_current():.6f} A")
    print(f"源功能: {gs.get_source_function()}")
else:
    print("未连接硬件，跳过")

电流设置为: 0.009290 A
源功能: CURR


---
## 3. 设置输出状态 (ON/OFF)

控制 GS200 输出继电器的通断。

In [8]:
if gs.connected:
    # 打开输出
    gs.set_output(True)
    print(f"输出状态: {'ON' if gs.get_output() else 'OFF'}")
    
    time.sleep(2)
    
    # 关闭输出
    gs.set_output(False)
    print(f"输出状态: {'ON' if gs.get_output() else 'OFF'}")
else:
    print("未连接硬件，跳过")

输出状态: ON
输出状态: OFF


---
## 4. 完整示例：输出指定电流

设置 → 打开输出 → 保持输出 → 关闭输出

In [ ]:
if gs.connected:
    print("=== GS200 电流输出示例 ===")
    
    # 设置电流 50 mA
    current_a = 0.05
    gs.set_current(current_a)
    print(f"1. 设置电流: {current_a*1000:.1f} mA")
    
    # 打开输出
    gs.set_output(True)
    print(f"2. 输出: ON")
    
    # 保持输出
    duration = 5
    print(f"3. 保持输出 {duration} 秒...")
    time.sleep(duration)
    
    # 关闭输出
    gs.set_output(False)
    print(f"4. 输出: OFF")
    print("=== 完成 ===")
else:
    print("未连接硬件，跳过")

---
## 5. 电压模式示例

将 GS200 切换为电压源模式并输出指定电压。

In [ ]:
if gs.connected:
    print("=== GS200 电压输出示例 ===")
    
    # 设置电压 5 V
    gs.set_voltage(5.0)
    print(f"1. 设置电压: {gs.get_voltage():.4f} V")
    print(f"   源功能: {gs.get_source_function()}")
    
    # 打开输出
    gs.set_output(True)
    print(f"2. 输出: ON")
    time.sleep(3)
    
    # 关闭输出
    gs.set_output(False)
    print(f"3. 输出: OFF")
    print("=== 完成 ===")
else:
    print("未连接硬件，跳过")

---
## 6. 保护设置

可设置限压和限流值以保护被测设备。

In [ ]:
if gs.connected:
    # 设置限压 6 V
    gs.set_voltage_limit(6.0)
    print(f"限压: {gs.get_voltage_limit():.4f} V")
    
    # 设置限流 200 mA
    gs.set_current_limit(0.2)
    print(f"限流: {gs.get_current_limit():.4f} A")
else:
    print("未连接硬件，跳过")

---
## 7. 断开连接

In [ ]:
if gs.connected:
    gs.disconnect()
    print("已断开 GS200 连接")
else:
    print("无活动连接")